In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image, ImageOps, ImageDraw, ImageFilter
import gradio as gr

C:\Users\locbo\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
num_classes = 10

In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224 )),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: 1.0 - x),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")

In [5]:
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
model.load_state_dict(torch.load('mobile_net.pth'))
model = model.to(device)
model.eval()

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [6]:

def crop_and_center_digit(img):
    """
    Hàm tự động cắt khung chữ số và căn giữa.

    Yêu cầu đầu vào: img là ảnh xám (L), nền trắng (255), nét đen (0).
    """
    mask = img.point(lambda p: 255 if p < 80 else 0)
    
    w, h = mask.size
    draw = ImageDraw.Draw(mask)
    draw.rectangle([0, 0, w, h], outline=0, width=5)
    
    bbox = mask.getbbox()
    
    if bbox is None:
        return img.resize((28, 28))
        
    cropped_img = img.crop(bbox)
    
    width, height = cropped_img.size
    max_dim = max(width, height)
    
    square_img = Image.new('L', (max_dim, max_dim), 255)
    paste_x = (max_dim - width) // 2
    paste_y = (max_dim - height) // 2
    square_img.paste(cropped_img, (paste_x, paste_y))
    
    square_img = square_img.resize((20, 20), Image.Resampling.BILINEAR)
    
    final_img = Image.new('L', (28, 28), 255)
    final_img.paste(square_img, (4, 4))
    
    return final_img

In [7]:
def predict(img):
    result = {}
    for i in range(10):
        result[str(i)] = 0
    img = img['composite']
    if img is None:
        return result
    
    img = img.convert('L')
        
    img = crop_and_center_digit(img)
    input_tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = torch.nn.functional.softmax(outputs[0], dim=0)

    for i in range(10):
        result[str(i)] = float(probabilities[i])
    return result

In [8]:
demo = gr.Interface(
    fn=predict,
    inputs=gr.Sketchpad(type="pil", label="Bảng vẽ"),
    
    outputs=gr.Label(num_top_classes=10, label="Xác suất dự đoán của MobileNet-V2"),
    live=True, 
    
    title="Nhận Diện Chữ Số Viết Tay",
    description="Vẽ một chữ số từ 0 đến 9. Bảng bên phải sẽ liên tục cập nhật suy nghĩ của mô hình."
)

if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
